In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


# Final publication analysis

This notebook implements cardinality-safe aggregation, five-fold outer/three-fold inner nested cross-validation, fold-specific maximum-F1 threshold selection, bootstrap confidence intervals, a proxy-resistant sensitivity analysis, publication figures, and an interactive dashboard.


## Imports, reproducibility settings, and Drive paths


In [2]:
from pathlib import Path
import json
import platform
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn

from sklearn.base import clone
from sklearn.ensemble import (
    ExtraTreesClassifier, GradientBoostingClassifier,
    HistGradientBoostingClassifier, RandomForestClassifier,
)
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, confusion_matrix, f1_score,
    precision_recall_curve, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
BOOTSTRAP_REPLICATES = 500

PROJECT = Path('/content/drive/MyDrive/Datasets/FYP')
candidate_data_dirs = [PROJECT, PROJECT / 'data', PROJECT / 'olist_data']

required_files = [
    'olist_customers_dataset.csv', 'olist_orders_dataset.csv',
    'olist_order_items_dataset.csv', 'olist_order_payments_dataset.csv',
    'olist_order_reviews_dataset.csv',
]
DATA = next(
    (folder for folder in candidate_data_dirs
     if all((folder / filename).exists() for filename in required_files)),
    None,
)
if DATA is None:
    raise FileNotFoundError(
        'Olist CSV files were not found. Put them in Datasets/FYP, '
        'Datasets/FYP/data, or Datasets/FYP/olist_data.'
    )

OUTPUT = PROJECT / 'Final_Publication_Results'
RESULTS_DIR = OUTPUT / 'results'
FIGURES_DIR = OUTPUT / 'figures'
MODELS_DIR = OUTPUT / 'models'
for folder in (RESULTS_DIR, FIGURES_DIR, MODELS_DIR):
    folder.mkdir(parents=True, exist_ok=True)


## Cardinality-safe relational aggregation and feature construction


In [3]:
customers = pd.read_csv(DATA / 'olist_customers_dataset.csv')
orders = pd.read_csv(
    DATA / 'olist_orders_dataset.csv',
    parse_dates=['order_purchase_timestamp', 'order_delivered_customer_date',
                 'order_estimated_delivery_date'],
)
items = pd.read_csv(DATA / 'olist_order_items_dataset.csv')
payments = pd.read_csv(DATA / 'olist_order_payments_dataset.csv')
reviews = pd.read_csv(DATA / 'olist_order_reviews_dataset.csv')

orders = (
    orders.loc[orders['order_status'].eq('delivered')]
    .merge(customers[['customer_id', 'customer_unique_id']], on='customer_id',
           how='left', validate='many_to_one')
)
item_agg = items.groupby('order_id', as_index=False).agg(
    num_items=('order_item_id', 'count'),
)
payment_rows = payments.assign(
    is_credit=payments['payment_type'].eq('credit_card').astype(float),
    is_boleto=payments['payment_type'].eq('boleto').astype(float),
)
payment_agg = payment_rows.groupby('order_id', as_index=False).agg(
    payment_total=('payment_value', 'sum'),
    avg_installments=('payment_installments', 'mean'),
    max_installments=('payment_installments', 'max'),
    credit_share=('is_credit', 'mean'),
    boleto_share=('is_boleto', 'mean'),
)
review_agg = reviews.groupby('order_id', as_index=False).agg(
    review_score_mean=('review_score', 'mean'),
)
order_master = (
    orders.merge(item_agg, on='order_id', how='left', validate='one_to_one')
    .merge(payment_agg, on='order_id', how='left', validate='one_to_one')
    .merge(review_agg, on='order_id', how='left', validate='one_to_one')
)
order_master['delivery_days'] = (
    order_master['order_delivered_customer_date']
    - order_master['order_purchase_timestamp']
).dt.total_seconds().div(86400)
delivery_known = (
    order_master['order_delivered_customer_date'].notna()
    & order_master['order_estimated_delivery_date'].notna()
)
order_master['late_delivery'] = np.where(
    delivery_known,
    (order_master['order_delivered_customer_date']
     > order_master['order_estimated_delivery_date']).astype(float),
    np.nan,
)
order_master['review_observed'] = order_master['review_score_mean'].notna().astype(float)
order_master['delivery_observed'] = order_master['delivery_days'].notna().astype(float)

reference_date = order_master['order_purchase_timestamp'].max() + pd.Timedelta(days=1)
customer_features = order_master.groupby('customer_unique_id', as_index=False).agg(
    frequency=('order_id', 'nunique'),
    monetary=('payment_total', 'sum'),
    avg_order_value=('payment_total', 'mean'),
    avg_items=('num_items', 'mean'),
    avg_installments=('avg_installments', 'mean'),
    max_installments=('max_installments', 'max'),
    credit_share=('credit_share', 'mean'),
    boleto_share=('boleto_share', 'mean'),
    avg_delivery_days=('delivery_days', 'mean'),
    late_delivery_rate=('late_delivery', 'mean'),
    avg_review_score=('review_score_mean', 'mean'),
    review_coverage=('review_observed', 'mean'),
    delivery_coverage=('delivery_observed', 'mean'),
    last_purchase=('order_purchase_timestamp', 'max'),
)
customer_features['recency_days'] = (
    reference_date - customer_features['last_purchase']
).dt.total_seconds().div(86400)
customer_features['repeat_purchase'] = customer_features['frequency'].ge(2).astype(int)

# Main retrospective ranking set. Monetary is retained as a deliberate
# historical-value signal, never combined with avg_order_value, so order count
# cannot be reconstructed algebraically. A proxy-resistant sensitivity analysis
# below removes all cumulative monetary information.
main_features = [
    'avg_review_score', 'avg_delivery_days', 'late_delivery_rate',
    'monetary', 'recency_days', 'avg_items', 'avg_installments',
    'max_installments', 'credit_share', 'boleto_share',
    'review_coverage', 'delivery_coverage',
]
sensitivity_features = [
    'avg_review_score', 'avg_delivery_days', 'late_delivery_rate',
    'avg_order_value', 'recency_days', 'avg_items', 'avg_installments',
    'credit_share', 'boleto_share', 'review_coverage', 'delivery_coverage',
]
y = customer_features['repeat_purchase'].reset_index(drop=True)


## Candidate pipelines, nested validation, metrics, and bootstrap functions


In [4]:
def model_candidates():
    return {
        'Logistic Regression': Pipeline([
            ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
            ('scaler', StandardScaler()),
            ('model', LogisticRegression(
                C=1.0, class_weight='balanced', max_iter=3000,
                random_state=SEED,
            )),
        ]),
        'Random Forest': Pipeline([
            ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
            ('model', RandomForestClassifier(
                n_estimators=400, min_samples_leaf=1, class_weight='balanced',
                random_state=SEED, n_jobs=1,
            )),
        ]),
        'Gradient Boosting': Pipeline([
            ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
            ('model', GradientBoostingClassifier(
                n_estimators=200, learning_rate=0.1, max_depth=3,
                random_state=SEED,
            )),
        ]),
        'Histogram Gradient Boosting': Pipeline([
            ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
            ('model', HistGradientBoostingClassifier(
                max_iter=200, learning_rate=0.1, max_leaf_nodes=15,
                class_weight='balanced', early_stopping=False,
                random_state=SEED,
            )),
        ]),
        'Extra Trees': Pipeline([
            ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
            ('model', ExtraTreesClassifier(
                n_estimators=400, min_samples_leaf=3, class_weight='balanced',
                random_state=SEED, n_jobs=1,
            )),
        ]),
    }

def maximum_f1_threshold(labels, scores):
    precision, recall, thresholds = precision_recall_curve(labels, scores)
    f1_values = 2 * precision[:-1] * recall[:-1] / (
        precision[:-1] + recall[:-1] + 1e-12
    )
    index = int(np.argmax(f1_values))
    return float(thresholds[index]), float(f1_values[index])


def lift_at_fraction(labels, scores, fraction=0.10):
    labels = np.asarray(labels)
    scores = np.asarray(scores)
    selected_count = int(np.ceil(fraction * len(labels)))
    selected = np.argsort(-scores, kind='mergesort')[:selected_count]
    return float(labels[selected].mean() / labels.mean())


def summarize_predictions(labels, percentile_score, prediction):
    tn, fp, fn, tp = confusion_matrix(labels, prediction).ravel()
    return {
        'customers': int(len(labels)),
        'positives': int(np.asarray(labels).sum()),
        'prevalence': float(np.asarray(labels).mean()),
        'accuracy': float(accuracy_score(labels, prediction)),
        'average_precision': float(average_precision_score(labels, percentile_score)),
        'roc_auc': float(roc_auc_score(labels, percentile_score)),
        'precision': float(precision_score(labels, prediction)),
        'recall': float(recall_score(labels, prediction)),
        'f1': float(f1_score(labels, prediction)),
        'lift_at_10_percent': lift_at_fraction(labels, percentile_score),
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
    }


def run_nested_analysis(feature_names, analysis_name):
    X = customer_features[feature_names].reset_index(drop=True)
    outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    nested_score = np.full(len(y), np.nan)
    nested_percentile = np.full(len(y), np.nan)
    nested_prediction = np.full(len(y), -1, dtype=int)
    nested_fold = np.full(len(y), -1, dtype=int)
    fold_records = []

    for fold, (outer_train, outer_test) in enumerate(outer_cv.split(X, y), start=1):
        X_train, y_train = X.iloc[outer_train], y.iloc[outer_train]
        inner_cv = StratifiedKFold(
            n_splits=3, shuffle=True, random_state=SEED + fold
        )
        inner_records = []
        inner_scores = {}
        for model_name, candidate in model_candidates().items():
            score = cross_val_predict(
                candidate, X_train, y_train, cv=inner_cv,
                method='predict_proba', n_jobs=-1,
            )[:, 1]
            ap = average_precision_score(y_train, score)
            roc = roc_auc_score(y_train, score)
            inner_records.append((model_name, ap, roc))
            inner_scores[model_name] = score

        selected_name, inner_ap, inner_roc = max(
            inner_records, key=lambda row: (row[1], row[2])
        )
        threshold, inner_f1 = maximum_f1_threshold(
            y_train, inner_scores[selected_name]
        )
        fitted = clone(model_candidates()[selected_name]).fit(X_train, y_train)
        outer_score = fitted.predict_proba(X.iloc[outer_test])[:, 1]
        outer_prediction = (outer_score >= threshold).astype(int)
        # Percentile normalization preserves within-fold ranking while making
        # scores comparable when different model families are selected.
        outer_percentile = pd.Series(outer_score).rank(
            method='average', pct=True
        ).to_numpy()
        fold_labels = y.iloc[outer_test]

        nested_score[outer_test] = outer_score
        nested_percentile[outer_test] = outer_percentile
        nested_prediction[outer_test] = outer_prediction
        nested_fold[outer_test] = fold
        fold_records.append({
            'analysis': analysis_name,
            'fold': fold,
            'selected_model': selected_name,
            'inner_ap': float(inner_ap),
            'inner_roc_auc': float(inner_roc),
            'inner_max_f1': inner_f1,
            'inner_max_f1_threshold': threshold,
            'outer_accuracy': accuracy_score(fold_labels, outer_prediction),
            'outer_average_precision': average_precision_score(
                fold_labels, outer_score
            ),
            'outer_roc_auc': roc_auc_score(fold_labels, outer_score),
            'outer_precision': precision_score(fold_labels, outer_prediction),
            'outer_recall': recall_score(fold_labels, outer_prediction),
            'outer_f1': f1_score(fold_labels, outer_prediction),
            'outer_lift_at_10_percent': lift_at_fraction(
                fold_labels, outer_score
            ),
        })
        print(
            'NESTED', analysis_name, 'FOLD', fold, fold_records[-1],
            flush=True,
        )

    assert np.isfinite(nested_score).all()
    assert np.isfinite(nested_percentile).all()
    assert (nested_prediction >= 0).all()
    summary = summarize_predictions(y, nested_percentile, nested_prediction)
    fold_frame = pd.DataFrame(fold_records)
    summary.update({
        'analysis': analysis_name,
        'feature_count': len(feature_names),
        'features': feature_names,
        'mean_outer_average_precision': float(
            fold_frame['outer_average_precision'].mean()
        ),
        'mean_outer_roc_auc': float(fold_frame['outer_roc_auc'].mean()),
        'mean_outer_lift_at_10_percent': float(
            fold_frame['outer_lift_at_10_percent'].mean()
        ),
        'fold_thresholds': fold_frame['inner_max_f1_threshold'].tolist(),
        'selected_models': fold_frame['selected_model'].tolist(),
    })
    return {
        'score': nested_score,
        'percentile_score': nested_percentile,
        'prediction': nested_prediction,
        'fold': nested_fold,
        'fold_metrics': fold_frame,
        'summary': summary,
    }


def stratified_bootstrap_intervals(labels, scores, predictions, replicates):
    labels = np.asarray(labels)
    scores = np.asarray(scores)
    predictions = np.asarray(predictions)
    positive = np.flatnonzero(labels == 1)
    negative = np.flatnonzero(labels == 0)
    rng = np.random.default_rng(SEED)
    records = []
    for _ in range(replicates):
        index = np.concatenate([
            rng.choice(positive, size=len(positive), replace=True),
            rng.choice(negative, size=len(negative), replace=True),
        ])
        rng.shuffle(index)
        record = summarize_predictions(
            labels[index], scores[index], predictions[index]
        )
        records.append(record)
    frame = pd.DataFrame(records)
    metrics = [
        'accuracy', 'average_precision', 'roc_auc', 'precision',
        'recall', 'f1', 'lift_at_10_percent',
    ]
    intervals = {}
    for metric in metrics:
        intervals[metric] = {
            'lower_95': float(frame[metric].quantile(0.025)),
            'upper_95': float(frame[metric].quantile(0.975)),
        }
    return intervals


## Execute primary and proxy-resistant nested analyses


In [5]:
main_nested = run_nested_analysis(main_features, 'main_retrospective')
sensitivity_nested = run_nested_analysis(
    sensitivity_features, 'proxy_resistant_sensitivity'
)
main_nested['summary']['bootstrap_95_ci'] = stratified_bootstrap_intervals(
    y, main_nested['percentile_score'], main_nested['prediction'],
    BOOTSTRAP_REPLICATES,
)
sensitivity_nested['summary']['bootstrap_95_ci'] = stratified_bootstrap_intervals(
    y, sensitivity_nested['percentile_score'], sensitivity_nested['prediction'],
    BOOTSTRAP_REPLICATES,
)


NESTED main_retrospective FOLD 1 {'analysis': 'main_retrospective', 'fold': 1, 'selected_model': 'Histogram Gradient Boosting', 'inner_ap': 0.7279931641935831, 'inner_roc_auc': 0.9326684026156081, 'inner_max_f1': 0.741266088783335, 'inner_max_f1_threshold': 0.8036948742261925, 'outer_accuracy': 0.9876820908311911, 'outer_average_precision': np.float64(0.7307312566657257), 'outer_roc_auc': np.float64(0.9366202557104998), 'outer_precision': 0.9319371727748691, 'outer_recall': 0.6357142857142857, 'outer_f1': 0.7558386411889597, 'outer_lift_at_10_percent': 7.81807892321811}
NESTED main_retrospective FOLD 2 {'analysis': 'main_retrospective', 'fold': 2, 'selected_model': 'Histogram Gradient Boosting', 'inner_ap': 0.7283851766061754, 'inner_roc_auc': 0.9350193528771877, 'inner_max_f1': 0.7363420427548611, 'inner_max_f1_threshold': 0.807484332371167, 'outer_accuracy': 0.987146529562982, 'outer_average_precision': np.float64(0.7423820702652676), 'outer_roc_auc': np.float64(0.9315923637840737), 

## Select and save the reusable production model and exact result tables


In [6]:
# Candidate comparison and the final deployable model are fitted only after
# nested performance estimation. These full-data OOF scores select the model
# family and threshold for future scoring; they are not the primary performance
# estimate reported by the paper.
X_main = customer_features[main_features].reset_index(drop=True)
selection_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
comparison_records = []
candidate_oof_scores = {}
for model_name, candidate in model_candidates().items():
    score = cross_val_predict(
        candidate, X_main, y, cv=selection_cv,
        method='predict_proba', n_jobs=-1,
    )[:, 1]
    candidate_oof_scores[model_name] = score
    comparison_records.append({
        'model': model_name,
        'oof_average_precision': average_precision_score(y, score),
        'oof_roc_auc': roc_auc_score(y, score),
    })
comparison_df = pd.DataFrame(comparison_records).sort_values(
    ['oof_average_precision', 'oof_roc_auc'], ascending=False
)
selected_model_name = comparison_df.iloc[0]['model']
production_oof_score = candidate_oof_scores[selected_model_name]
production_threshold, production_oof_f1 = maximum_f1_threshold(
    y, production_oof_score
)
production_model = clone(model_candidates()[selected_model_name]).fit(X_main, y)
joblib.dump(
    {
        'model': production_model,
        'features': main_features,
        'threshold': production_threshold,
        'scope': 'retrospective observed-repeat-status ranking',
        'score_is_calibrated_probability': False,
    },
    MODELS_DIR / 'final_model_bundle.joblib',
)

main_summary = main_nested['summary']
sensitivity_summary = sensitivity_nested['summary']
final_results = {
    'scope': 'retrospective observed-repeat-status ranking',
    'validation': (
        'five-fold outer nested cross-validation; three-fold inner model '
        'selection and maximum-F1 threshold selection'
    ),
    'primary_nested_results': main_summary,
    'proxy_resistant_sensitivity_results': sensitivity_summary,
    'production_model': {
        'selected_model': selected_model_name,
        'full_data_oof_threshold': production_threshold,
        'full_data_oof_f1_at_threshold': production_oof_f1,
        'performance_claim_source': 'nested cross-validation, not full-data OOF',
    },
}
with open(RESULTS_DIR / 'final_publication_metrics.json', 'w', encoding='utf-8') as handle:
    json.dump(final_results, handle, indent=2)

main_nested['fold_metrics'].to_csv(
    RESULTS_DIR / 'nested_fold_metrics_main.csv', index=False
)
sensitivity_nested['fold_metrics'].to_csv(
    RESULTS_DIR / 'nested_fold_metrics_sensitivity.csv', index=False
)
comparison_df.to_csv(RESULTS_DIR / 'model_comparison_oof.csv', index=False)

customer_output = customer_features[
    ['customer_unique_id', 'repeat_purchase'] + sorted(
        set(main_features + sensitivity_features)
    )
].copy()
customer_output['nested_raw_score'] = main_nested['score']
customer_output['nested_percentile_score'] = main_nested['percentile_score']
customer_output['nested_repeat_flag'] = main_nested['prediction']
customer_output['outer_fold'] = main_nested['fold']
customer_output['score_segment'] = pd.qcut(
    customer_output['nested_percentile_score'].rank(method='first'),
    q=3, labels=['Low', 'Medium', 'High'],
)
customer_output.to_csv(
    RESULTS_DIR / 'customer_nested_cross_fitted_scores.csv', index=False
)

environment = pd.DataFrame({
    'component': [
        'Python', 'platform', 'NumPy', 'pandas', 'scikit-learn',
        'Matplotlib', 'joblib',
    ],
    'version': [
        sys.version.split()[0], platform.platform(), np.__version__,
        pd.__version__, sklearn.__version__, plt.matplotlib.__version__,
        joblib.__version__,
    ],
})
environment.to_csv(RESULTS_DIR / 'environment_versions.csv', index=False)
pd.DataFrame({
    'feature': main_features + sensitivity_features,
    'analysis': (
        ['main_retrospective'] * len(main_features)
        + ['proxy_resistant_sensitivity'] * len(sensitivity_features)
    ),
}).to_csv(RESULTS_DIR / 'feature_manifest.csv', index=False)


## Generate all publication figures and the interactive dashboard function


In [7]:
# ---------------------------------------------------------------------------
# Exact publication figures
# ---------------------------------------------------------------------------
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 10,
    'axes.titlesize': 13, 'axes.labelsize': 10,
    'figure.dpi': 120, 'savefig.dpi': 300,
})
blue, dark_blue = '#1F77B4', '#174A7E'
orange, green, red = '#F28E2B', '#2CA06C', '#C8443A'
light_blue = '#EAF3F8'

# Figure 1: nested workflow.
fig, ax = plt.subplots(figsize=(13, 4.8))
ax.axis('off')
workflow = [
    ('Relational\naggregation', 'Order-level joins\nwithout duplication'),
    ('Feature sets', 'Main retrospective +\nproxy-resistant sensitivity'),
    ('Outer 5-fold CV', 'Every customer held\nout exactly once'),
    ('Inner 3-fold CV', 'Select model +\nmaximum-F1 threshold'),
    ('Evidence + BI', 'Nested metrics, CIs,\nlift, dashboard'),
]
x_positions = np.linspace(.02, .82, len(workflow))
for index, ((title, subtitle), x) in enumerate(zip(workflow, x_positions), start=1):
    ax.add_patch(plt.Rectangle(
        (x, .33), .16, .36, facecolor=light_blue,
        edgecolor=blue, linewidth=2, transform=ax.transAxes,
    ))
    ax.text(x + .08, .57, f'{index}. {title}', ha='center', va='center',
            fontsize=10.5, fontweight='bold', transform=ax.transAxes)
    ax.text(x + .08, .42, subtitle, ha='center', va='center',
            fontsize=8.7, color='#444444', transform=ax.transAxes)
for left, right in zip(x_positions[:-1], x_positions[1:]):
    ax.annotate('', xy=(right, .51), xytext=(left + .16, .51),
                xycoords=ax.transAxes, textcoords=ax.transAxes,
                arrowprops=dict(arrowstyle='->', color=dark_blue, lw=2))
ax.text(.5, .16,
        'Primary performance comes from outer held-out folds; no single reused test set',
        ha='center', fontsize=11, color=dark_blue, fontweight='bold',
        transform=ax.transAxes)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'Figure_1_Nested_Methodology.png', bbox_inches='tight')
plt.close(fig)

# Figure 2: fixed-candidate OOF comparison used for final production selection.
plot_comparison = comparison_df.sort_values('oof_average_precision')
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
axes[0].barh(plot_comparison['model'], plot_comparison['oof_roc_auc'], color=blue)
axes[1].barh(plot_comparison['model'], plot_comparison['oof_average_precision'], color=orange)
axes[0].set(title='Cross-Fitted ROC-AUC', xlim=(0, 1))
axes[1].set(title='Cross-Fitted Average Precision', xlim=(0, 1))
for axis, metric in zip(axes, ['oof_roc_auc', 'oof_average_precision']):
    for row, value in enumerate(plot_comparison[metric]):
        axis.text(value + .012, row, f'{value:.3f}', va='center', fontsize=9)
fig.suptitle('Model Comparison for Final Production Selection', fontweight='bold')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'Figure_2_Model_Comparison.png', bbox_inches='tight')
plt.close(fig)

# Figure 3: nested cross-validated PR curve on fold-normalized scores.
curve_p, curve_r, _ = precision_recall_curve(
    y, main_nested['percentile_score']
)
fig, ax = plt.subplots(figsize=(6.2, 5.8))
ax.plot(curve_r, curve_p, color=blue, lw=2.6,
        label=f'Nested CV (AP={main_summary["average_precision"]:.3f})')
ax.axhline(y.mean(), color='#666666', ls='--',
           label=f'Prevalence={y.mean():.3f}')
ax.set(xlabel='Recall', ylabel='Precision', xlim=(0, 1), ylim=(0, 1),
       title='Nested Cross-Validated Precision-Recall Curve')
ax.legend(loc='upper right', frameon=True)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'Figure_3_PR_Curve.png', bbox_inches='tight')
plt.close(fig)

# Figure 4: pooled outer-fold confusion matrix with inner-selected thresholds.
cm = np.array([
    [main_summary['tn'], main_summary['fp']],
    [main_summary['fn'], main_summary['tp']],
])
fig, ax = plt.subplots(figsize=(5.8, 5.2))
image = ax.imshow(cm, cmap='Blues')
for row in range(2):
    for column in range(2):
        ax.text(column, row, f'{cm[row, column]:,}', ha='center', va='center',
                fontsize=15, fontweight='bold',
                color='white' if cm[row, column] > cm.max()/2 else 'black')
ax.set_xticks([0, 1], ['Predicted one-time', 'Predicted repeat'])
ax.set_yticks([0, 1], ['Observed one-time', 'Observed repeat'])
ax.set_title('Nested-CV Confusion Matrix\n(inner maximum-F1 thresholds)')
fig.colorbar(image, ax=ax, fraction=.046)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'Figure_4_Confusion_Matrix.png', bbox_inches='tight')
plt.close(fig)

# Figure 5: final fitted-model permutation importance (descriptive, not causal).
# This model-agnostic method works for every candidate estimator, including HGB.
permutation = permutation_importance(
    production_model,
    X_main,
    y,
    scoring='average_precision',
    n_repeats=5,
    random_state=SEED,
    n_jobs=-1,
)
importance_df = pd.DataFrame({
    'feature': main_features,
    'importance': permutation.importances_mean,
    'importance_sd': permutation.importances_std,
}).sort_values('importance', ascending=False)
importance_df.to_csv(RESULTS_DIR / 'feature_importance.csv', index=False)
fig, ax = plt.subplots(figsize=(8, 6))
top_importance = importance_df.head(15).sort_values('importance')
ax.barh(top_importance['feature'], top_importance['importance'], color=green)
ax.axvline(0, color='black', linewidth=.8)
ax.set(xlabel='Decrease in average precision after permutation',
       title=f'Final {selected_model_name} Permutation Importance')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'Figure_5_Feature_Importance.png', bbox_inches='tight')
plt.close(fig)

# Figure 6: nested cross-fitted score segments.
segment_rates = customer_output.groupby('score_segment', observed=True).agg(
    customers=('customer_unique_id', 'size'),
    observed_repeat_rate=('repeat_purchase', 'mean'),
).reset_index()
segment_rates.to_csv(RESULTS_DIR / 'nested_segment_rates.csv', index=False)
fig, ax = plt.subplots(figsize=(7.4, 5.2))
bars = ax.bar(
    segment_rates['score_segment'].astype(str),
    100 * segment_rates['observed_repeat_rate'],
    color=[blue, orange, green],
)
for bar, rate in zip(bars, segment_rates['observed_repeat_rate']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + .12,
            f'{100*rate:.2f}%', ha='center', fontweight='bold')
ax.set(xlabel='Nested cross-fitted score segment',
       ylabel='Observed repeat rate (%)',
       title='Observed Repeat Rate by Score Segment')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'Figure_6_Customer_Segments.png', bbox_inches='tight')
plt.close(fig)

# Figure 7: nested cross-fitted lift deciles.
lift_frame = pd.DataFrame({
    'score': main_nested['percentile_score'], 'target': y,
})
lift_frame['decile'] = pd.qcut(
    lift_frame['score'].rank(method='first', ascending=False),
    10, labels=range(1, 11),
)
lift_table = lift_frame.groupby('decile', observed=True).agg(
    customers=('target', 'size'), positives=('target', 'sum'),
    positive_rate=('target', 'mean'),
).reset_index()
lift_table['lift'] = lift_table['positive_rate'] / y.mean()
lift_table.to_csv(RESULTS_DIR / 'nested_lift_deciles.csv', index=False)
fig, ax = plt.subplots(figsize=(8.2, 5.2))
bars = ax.bar(lift_table['decile'].astype(int), lift_table['lift'], color=blue)
ax.axhline(1, color=red, ls='--', lw=1.8, label='Population baseline')
ax.text(bars[0].get_x() + bars[0].get_width()/2, bars[0].get_height() + .1,
        f'{bars[0].get_height():.2f}×', ha='center', fontweight='bold')
ax.set(xlabel='Score decile (1 = highest)', ylabel='Lift',
       title='Nested Cross-Fitted Lift by Decile')
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'Figure_7_Lift_Deciles.png', bbox_inches='tight')
plt.close(fig)

# Figure 9: static evidence view corresponding to the interactive dashboard.
fig = plt.figure(figsize=(13.5, 7.5), facecolor='white')
grid = fig.add_gridspec(2, 4, height_ratios=[1, 2.1], hspace=.35, wspace=.28)
kpis = [
    ('Customers', f'{len(y):,}'),
    ('Nested ROC-AUC', f'{main_summary["roc_auc"]:.3f}'),
    ('Nested AP', f'{main_summary["average_precision"]:.3f}'),
    ('Lift@10%', f'{main_summary["lift_at_10_percent"]:.2f}×'),
]
for column, (label, value) in enumerate(kpis):
    axis = fig.add_subplot(grid[0, column])
    axis.axis('off')
    axis.add_patch(plt.Rectangle(
        (.02, .08), .96, .84, transform=axis.transAxes,
        facecolor=light_blue, edgecolor=blue, linewidth=1.8,
    ))
    axis.text(.5, .62, value, transform=axis.transAxes, ha='center',
              va='center', fontsize=23, fontweight='bold', color=dark_blue)
    axis.text(.5, .29, label, transform=axis.transAxes, ha='center',
              va='center', fontsize=10.5, color='#333333')
left = fig.add_subplot(grid[1, :2])
left.bar(segment_rates['score_segment'].astype(str),
         100 * segment_rates['observed_repeat_rate'],
         color=[blue, orange, green])
left.set(title='Cross-Fitted Customer Segments',
         ylabel='Observed repeat rate (%)')
right = fig.add_subplot(grid[1, 2:])
right.bar(lift_table['decile'].astype(int), lift_table['lift'], color=blue)
right.axhline(1, color=red, ls='--')
right.set(title='Nested Cross-Fitted Lift', xlabel='Decile (1 = highest)',
          ylabel='Lift')
fig.suptitle('Repeat-Purchase Ranking: Auditable BI Dashboard',
             fontsize=17, fontweight='bold', color=dark_blue)
fig.tight_layout(rect=[0, 0, 1, .95])
fig.savefig(FIGURES_DIR / 'Figure_9_BI_Dashboard.png', bbox_inches='tight')
plt.close(fig)

# Figure 8: robustness comparison.
robustness_df = pd.DataFrame([
    {
        'analysis': 'Main retrospective',
        'ROC-AUC': main_summary['roc_auc'],
        'AP': main_summary['average_precision'],
        'F1': main_summary['f1'],
        'Lift@10%': main_summary['lift_at_10_percent'],
    },
    {
        'analysis': 'Without cumulative monetary',
        'ROC-AUC': sensitivity_summary['roc_auc'],
        'AP': sensitivity_summary['average_precision'],
        'F1': sensitivity_summary['f1'],
        'Lift@10%': sensitivity_summary['lift_at_10_percent'],
    },
])
robustness_df.to_csv(RESULTS_DIR / 'robustness_comparison.csv', index=False)
fig, axes = plt.subplots(1, 4, figsize=(13, 4.2))
for axis, metric in zip(axes, ['ROC-AUC', 'AP', 'F1', 'Lift@10%']):
    values = robustness_df[metric]
    axis.bar(['Main', 'No cumulative\nmonetary'], values,
             color=[blue, orange])
    axis.set_title(metric)
    if metric != 'Lift@10%':
        axis.set_ylim(0, 1)
    for index, value in enumerate(values):
        axis.text(index, value + (.02 if metric != 'Lift@10%' else .08),
                  f'{value:.3f}' if metric != 'Lift@10%' else f'{value:.2f}×',
                  ha='center', fontweight='bold', fontsize=9)
fig.suptitle('Proxy-Resistance Sensitivity Analysis', fontweight='bold')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'Figure_8_Robustness_Analysis.png', bbox_inches='tight')
plt.close(fig)


def launch_interactive_dashboard(data):
    """Launch a filter-responsive Colab/Jupyter research dashboard."""
    import ipywidgets as widgets
    from IPython.display import HTML, clear_output, display

    segment = widgets.Dropdown(
        options=['All', 'Low', 'Medium', 'High'], value='All',
        description='Segment:',
    )
    recency = widgets.IntRangeSlider(
        value=[int(data['recency_days'].min()), int(data['recency_days'].max())],
        min=int(data['recency_days'].min()), max=int(data['recency_days'].max()),
        description='Recency:', continuous_update=False,
    )
    score = widgets.FloatRangeSlider(
        value=[0.0, 1.0], min=0.0, max=1.0, step=0.01,
        description='Score:', readout_format='.2f', continuous_update=False,
    )
    monetary = widgets.FloatRangeSlider(
        value=[float(data['monetary'].min()), float(data['monetary'].max())],
        min=float(data['monetary'].min()), max=float(data['monetary'].max()),
        step=10.0, description='Monetary:', readout_format='.0f',
        continuous_update=False,
    )
    output = widgets.Output()

    def refresh(*_):
        filtered = data.loc[
            data['recency_days'].between(recency.value[0], recency.value[1])
            & data['nested_percentile_score'].between(score.value[0], score.value[1])
            & data['monetary'].between(monetary.value[0], monetary.value[1])
        ].copy()
        if segment.value != 'All':
            filtered = filtered.loc[filtered['score_segment'].astype(str).eq(segment.value)]
        with output:
            clear_output(wait=True)
            if filtered.empty:
                display(HTML('<b>No customers match the current filters.</b>'))
                return
            labels = filtered['repeat_purchase']
            scores = filtered['nested_percentile_score']
            kpis = [f'Customers: {len(filtered):,}']
            if labels.nunique() == 2:
                kpis.extend([
                    f'Filtered cross-fitted ROC-AUC: {roc_auc_score(labels, scores):.3f}',
                    f'Filtered cross-fitted AP: {average_precision_score(labels, scores):.3f}',
                ])
            display(HTML('<h3>' + ' &nbsp; | &nbsp; '.join(kpis) + '</h3>'))
            operational_columns = [
                'customer_unique_id', 'score_segment',
                'nested_percentile_score', 'nested_repeat_flag',
                'recency_days', 'avg_review_score', 'avg_delivery_days',
            ]
            display(
                filtered.sort_values('nested_percentile_score', ascending=False)
                [operational_columns].head(20)
            )

    segment.observe(refresh, names='value')
    recency.observe(refresh, names='value')
    score.observe(refresh, names='value')
    monetary.observe(refresh, names='value')
    refresh()
    display(widgets.VBox([
        widgets.HBox([segment, recency]),
        widgets.HBox([score, monetary]),
        output,
    ]))


/tmp/ipykernel_1451/2307978746.py:205: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout(rect=[0, 0, 1, .95])


## Run independent checks and print final results


In [8]:
# Independent checks.
expected_figures = [
    'Figure_1_Nested_Methodology.png', 'Figure_2_Model_Comparison.png',
    'Figure_3_PR_Curve.png', 'Figure_4_Confusion_Matrix.png',
    'Figure_5_Feature_Importance.png', 'Figure_6_Customer_Segments.png',
    'Figure_7_Lift_Deciles.png', 'Figure_8_Robustness_Analysis.png',
    'Figure_9_BI_Dashboard.png',
]
assert len(customer_features) == 93358
assert int(y.sum()) == 2801
assert 'frequency' not in main_features
assert 'avg_order_value' not in main_features
assert 'monetary' not in sensitivity_features
assert sum(main_summary[key] for key in ['tn', 'fp', 'fn', 'tp']) == len(y)
assert all((FIGURES_DIR / filename).exists() for filename in expected_figures)
assert customer_output['customer_unique_id'].notna().all()

print('\nFINAL PUBLICATION RESULTS')
print(json.dumps(final_results, indent=2))
print('\nAll independent checks passed.')
print('Results saved in:', RESULTS_DIR)
print('Figures saved in:', FIGURES_DIR)
print('Model saved in:', MODELS_DIR)



FINAL PUBLICATION RESULTS
{
  "scope": "retrospective observed-repeat-status ranking",
  "validation": "five-fold outer nested cross-validation; three-fold inner model selection and maximum-F1 threshold selection",
  "primary_nested_results": {
    "customers": 93358,
    "positives": 2801,
    "prevalence": 0.03000278497825575,
    "accuracy": 0.9874354634846505,
    "average_precision": 0.7397291352623296,
    "roc_auc": 0.9377642431342946,
    "precision": 0.9094567404426559,
    "recall": 0.6454837558014994,
    "f1": 0.7550636876174567,
    "lift_at_10_percent": 7.964841865449571,
    "tn": 90377,
    "fp": 180,
    "fn": 993,
    "tp": 1808,
    "analysis": "main_retrospective",
    "feature_count": 12,
    "features": [
      "avg_review_score",
      "avg_delivery_days",
      "late_delivery_rate",
      "monetary",
      "recency_days",
      "avg_items",
      "avg_installments",
      "max_installments",
      "credit_share",
      "boleto_share",
      "review_coverage",
 